# 03 — v3 lambda Sweep (structural hub, PDF Full model)

Trains v3 (nutrition hub nodes + I-N edges) across lambda in {0, 0.1, 1, 5, 10}. lambda=0 is automatically the `w/o L_health` ablation. Produces `sweep_summary_v3.csv` for the Pareto curve in `06_eval_results.ipynb`.

**Runtime**: Colab free T4 GPU, ~2.5 hr.

**Steps**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell 5) to wherever you put `data/` in your Drive
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# torch_geometric needs to be matched with the installed torch wheel.
# Colab's default torch is recent enough that the generic install works.
!pip install -q torch_geometric pandas numpy matplotlib

## 2. Get the code

In [ ]:
# Clone (or refresh) the repo
import os
if not os.path.exists('/content/CS471-Project'):
    !git clone https://github.com/lee1june61/CS471-Project.git /content/CS471-Project
else:
    !cd /content/CS471-Project && git pull --ff-only
%cd /content/CS471-Project

## 3. Mount Drive for data + outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === EDIT THIS if your Drive layout is different ===
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# sanity check
expected = ['flavorgraph_edges.csv', 'nodes_filtered.csv',
             'pairs_train.csv', 'pairs_val.csv', 'pairs_test.csv',
             'recipes.json', 'usda_mapping.json']
missing = [f for f in expected if not os.path.exists(f'{DATA_DIR}/{f}')]
if missing:
    raise FileNotFoundError(f'missing data files in {DATA_DIR}: {missing}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 4. Smoke test (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Skip by changing False below.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v3.py --max_epochs 1 --patience 1 \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v3
    print('\n[smoke] OK')

## 5. Run lambda sweep

In [ ]:
# Sweep produces per-lambda subdir + sweep_summary_v3.csv
!python src/run_lambda_sweep.py --variant v3 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} \
  --tau_percentile 0 \
  --lambdas 0 0.1 1 5 10

## 6. Quick look at the sweep summary

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/sweep_summary_v3.csv')
show = ['lambda_h', 'MRR', 'Hit@10', 'sugar_sat_pct',
         'sodium_sat_pct', 'flavor_cos_mean']
print(df[[c for c in show if c in df.columns]].round(2).to_string(index=False))

## 7. (Optional) Re-evaluate best-lambda v3 with all 4 g overrides

Needed for the g-override sensitivity check and case study in `06_eval_results.ipynb`. Re-uses the saved checkpoint (no retraining).

In [ ]:
BEST_LAMBDA = 1.0  # change after looking at the sweep table above
best_tag = str(BEST_LAMBDA).replace('.', '_')
best_dir = f'{OUTPUT_DIR}/v3_lam{best_tag}'

!python src/train_v3.py --lambda_h {BEST_LAMBDA} --tau_percentile 0 \
  --test_g_overrides auto 1_0 0_1 1_1 \
  --data_dir {DATA_DIR} --output_dir {best_dir}

print(f'\n[done] g-override predictions saved under {best_dir}/')